# Data Cleaning and Chronological Split

This notebook prepares the IPEDS panel for feature engineering. It loads the source data, removes deterministic exclusions and leakage, builds the labeled modeling table, and creates a forward-year train/validation/test split.

## 0. Imports and settings

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

C:\Users\jimmy\miniconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


**Setup.** Imports pandas and numpy and widens the display limits so the wide IPEDS tables print in full while inspecting.

## 1. Load the data

In [2]:
df = pd.read_csv("../Data/master_panel_df.csv", low_memory=False)

# Some yearly files carried trailing spaces in a few headers (e.g. "SFTEPTMM   "),
# which the concat left as separate columns. Strip the names and keep the first.
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.duplicated()]

print("shape:", df.shape)
df.head()

shape: (8670, 540)


,UNITID,INSTNM,ADDR,CITY,STABBR,ZIP,FIPS,OBEREG,CHFNM,CHFTITLE,GENTELE,EIN,OPEID,OPEFLAG,WEBADDR,ADMINURL,FAIDURL,APPLURL,NPRICURL,VETURL,ATHURL,SECTOR,ICLEVEL,CONTROL,HLOFFER,UGOFFER,GROFFER,HDEGOFR1,DEGGRANT,HBCU,HOSPITAL,MEDICAL,TRIBAL,LOCALE,OPENPUBL,ACT,NEWID,DEATHYR,CLOSEDAT,CYACTIVE,POSTSEC,PSEFLAG,PSET4FLG,RPTMTH,IALIAS,INSTCAT,CCBASIC,CCIPUG,CCIPGRAD,CCUGPROF,CCENRPRF,CCSIZSET,CARNEGIE,LANDGRNT,INSTSIZE,CBSA,CBSATYPE,CSA,NECTA,F1SYSTYP,F1SYSNAM,F1SYSCOD,COUNTYCD,COUNTYNM,CNGDSTCD,LONGITUD,LATITUDE,DFRCGID,DFRCUSCG,XF1A01,F1A01,XF1A31,F1A31,XF1A04,F1A04,XF1A05,F1A05,XF1A06,F1A06,XF1A07,F1A07,XF1A08,F1A08,XF1A09,F1A09,XF1A10,F1A10,XF1A11,F1A11,XF1A12,F1A12,XF1A13,F1A13,XF1A14,F1A14,XF1A15,F1A15,XF1A16,F1A16,XF1A17,...,XF1C19DP,F1C19DP,XF1C19IN,F1C19IN,XF1C19OT,F1C19OT,C18BASIC,C18IPUG,C18IPGRD,C18UGPRF,C18ENPRF,C18SZSET,XF1M05,F1M05,XF1M06,F1M06,XF1M07,F1M07,XF1M08,F1M08,F1MHOP,XF1E12,F1E12,XF1E121,F1E121,XF1E122,F1E122,XF1E13,F1E13,XF1E131,F1E131,XF1E132,F1E132,XF1E14,F1E14,XF1E141,F1E141,XF1E142,F1E142,XF1E15,F1E15,XF1E151,F1E151,XF1E152,F1E152,XF1E16,F1E16,XF1E161,F1E161,XF1E162,F1E162,XF1E17,F1E17,XF1E171,F1E171,XF1E172,F1E172,XF1H03,F1H03,XF1H03A,F1H03A,XF1H03B,F1H03B,XF1H03C,F1H03C,XF1H03D,F1H03D,XF1N01,F1N01,XF1N02,F1N02,XF1N03,F1N03,XF1N04,F1N04,XF1N05,F1N05,XF1N06,F1N06,XF1N07,F1N07,C21BASIC,C21IPUG,C21IPGRD,C21UGPRF,C21ENPRF,C21SZSET,UEIS,C00CARNEGIE,CARNEGIEIC,CARNEGIESAEC,CARNEGIERSCH,CARNEGIESIZE,CARNEGIEALF,CARNEGIEAPM,CARNEGIEGPM,FTE_INST_PCT_CHANGE_2Y,FTE_TOTL_PCT_CHANGE_2Y,TARGET_STAFF_CUT,TARGET_VULNERABLE_TOTAL
0,100654,Alabama A & M University,4900 Meridian Street,Normal,AL,35762,1,5,"Dr. Andrew Hugine, Jr.",President,2563725000,636001109,100200.0,1,www.aamu.edu/,www.aamu.edu/admissions/pages/default.aspx,www.aamu.edu/Admissions/fincialaid/Pages/defau...,www.aamu.edu/Admissions/apply/Pages/default.aspx,galileo.aamu.edu/netpricecalculator/npcalc.htm,,www.aamu.edu,1,1,1,9,1,1,12,1,1,2,2,2,12,1,A,-2,-2,-2,1,1,1,1,1,AAMU,2,18.0,13.0,18.0,9.0,4.0,14.0,16.0,1,3,26620,1,290,-2.0,2,,-2,1089,Madison,105,-86.568502,34.783368,131,1,R,76015628.0,R,103355825.0,R,3979751.0,R,107335576.0,R,183351204.0,R,3420000.0,R,52416654.0,R,55836654.0,R,46351285.0,R,0.0,R,46351285.0,R,102187939.0,R,52646385.0,R,17608547.0,R,0.0,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.599190,6.549118,0.0,0.0
1,100654,Alabama A & M University,4900 Meridian Street,Normal,AL,35762,1,5,"Dr. Andrew Hugine, Jr.",President,2563725000,636001109,100200.0,1,www.aamu.edu/,www.aamu.edu/admissions/pages/default.aspx,www.aamu.edu/Admissions/fincialaid/Pages/defau...,www.aamu.edu/admissions/undergraduateadmission...,galileo.aamu.edu/netpricecalculator/npcalc.htm,,www.aamu.edu,1,1,1,9,1,1,12,1,1,2,2,2,12,1,A,-2,-2,-2,1,1,1,1,1,AAMU,2,18.0,NaN,NaN,NaN,NaN,NaN,16.0,1,3,26620,1,290,-2.0,2,,-2,1089,Madison County,105,-86.568502,34.783368,135,1,R,78535712.0,R,115726368.0,R,13423.0,R,115739791.0,R,194275503.0,R,1623631.0,R,62725127.0,R,64348758.0,R,60180723.0,R,65496062.0,R,125676785.0,R,190025543.0,R,57082083.0,R,639990.0,R,0.0,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.659498,3.485577,0.0,0.0
2,100654,Alabama A & M University,4900 Meridian Street,Normal,AL,35762,1,5,"Dr. Andrew Hugine, Jr.",President,2563725000,636001109,100200.0,1,www.aamu.edu/,www.aamu.edu/Admissions/Pages/d

## 2. Configuration (AI-Assisted)

In [3]:
TARGET = "TARGET_STAFF_CUT"
KEYS = ["UNITID", "YEAR"]

# Columns computed with t+1 information or representing another forward label.
LEAKAGE = [
    "FTE_INST_PCT_CHANGE",
    "FTE_TOTL_PCT_CHANGE",
    "TARGET_VULNERABLE_TOTAL",
]

# Identifiers, administrative fields, redundant geography, high-cardinality
# codes, and overlapping institutional classifications.
ADMIN = [
    # Institution names, addresses, contacts, and identifiers
    "INSTNM",
    "IALIAS",
    "ADDR",
    "CITY",
    "ZIP",
    "LATITUDE",
    "LONGITUD",
    "CHFNM",
    "CHFTITLE",
    "GENTELE",
    "EIN",
    "OPEID",
    "DUNS",
    "UEIS",

    # URLs
    "WEBADDR",
    "ADMINURL",
    "FAIDURL",
    "APPLURL",
    "NPRICURL",
    "VETURL",
    "ATHURL",
    "DISAURL",

    # Names and dates
    "COUNTYNM",
    "F1SYSNAM",
    "CLOSEDAT",

    # Administrative status and reporting fields
    "ACT",
    "OPEFLAG",
    "PSET4FLG",
    "OPENPUBL",
    "PSEFLAG",
    "RPTMTH",

    # Redundant or high-cardinality geography
    "FIPS",
    "COUNTYCD",
    "CBSA",
    "CSA",
    "NECTA",
    "OBEREG",

    # High-cardinality institution/system identifiers
    "F1SYSCOD",
    "DFRCGID",
    "DFRCUSCG",

    # Overlapping institutional classifications
    "CARNEGIE",
    "HLOFFER",
    "UGOFFER",
    "GROFFER",
]

# Columns known to contain only one value in the modeling extract.
KNOWN_CONSTANTS = [
    "DEGGRANT",
    "NEWID",
    "DEATHYR",
    "CYACTIVE",
]

# Classification vintages that may introduce future information.
TIME_SENSITIVE_CLASSIFICATIONS = [
    "C15BASIC",
    "C18BASIC",
]

# Numeric IPEDS codes that remain in the model and should be treated
# as categorical labels rather than continuous measurements.
CATEGORICAL_CODES = [
    "HBCU",
    "HOSPITAL",
    "MEDICAL",
    "TRIBAL",
    "LOCALE",
    "INSTCAT",
    "CCBASIC",
    "LANDGRNT",
    "INSTSIZE",
    "CBSATYPE",
    "F1SYSTYP",
    "F1FHA",
    "F1MHP",
    "HDEGOFR1",
]

**Configuration.** Defines the target and key columns, then the groups to handle specially: `LEAKAGE` (computed from *t+1*, so they'd leak the answer), `ADMIN` (identifiers, URLs, and redundant geography with no predictive value), `KNOWN_CONSTANTS` (single-value columns), `TIME_SENSITIVE_CLASSIFICATIONS` (classification vintages that can carry future information), and `CATEGORICAL_CODES` (numeric IPEDS codes that are really labels, not measurements).

### Administrative and excluded feature definitions

* `INSTNM`: Institution name
* `IALIAS`: Alternative or commonly used institution name
* `ADDR`, `CITY`, `ZIP`: Institution mailing-address fields
* `LATITUDE`, `LONGITUD`: Institution geographic coordinates
* `CHFNM`, `CHFTITLE`: Chief administrator name and title
* `GENTELE`: General institution telephone number
* `EIN`, `OPEID`, `DUNS`, `UEIS`: Institutional, tax, federal-aid, and federal-entity identifiers
* `WEBADDR`, `ADMINURL`, `FAIDURL`, `APPLURL`, `NPRICURL`, `VETURL`, `ATHURL`, `DISAURL`: Institution, admissions, financial-aid, application, net-price, veteran, athletics, and disability-services website fields
* `COUNTYNM`: County name
* `F1SYSNAM`: Name of the governing university or institutional system
* `CLOSEDAT`: Institution closure date

### Administrative status and reporting fields

* `ACT`: Institution activity status. The modeling sample is first restricted to active institutions, after which this field has no remaining variation.
* `OPEFLAG`: Office of Postsecondary Education eligibility or participation status
* `PSET4FLG`: Postsecondary institution and Title IV participation classification
* `OPENPUBL`: Indicator related to whether the institution is open to the public
* `PSEFLAG`: Postsecondary institution status indicator
* `RPTMTH`: IPEDS reporting-method classification

### Geographic fields excluded from the primary model

* `FIPS`: State or jurisdiction Federal Information Processing Standards code
* `COUNTYCD`: County geographic code
* `CBSA`: Core-Based Statistical Area code identifying a metropolitan or micropolitan area
* `CSA`: Combined Statistical Area code grouping economically connected metropolitan and micropolitan areas
* `NECTA`: New England City and Town Area code
* `OBEREG`: Broad geographic region classification

`NECTA` is converted into `HAS_NECTA`, a binary indicator showing whether an institution belongs to a valid NECTA area, before the original code is removed.

These geographic codes are excluded because they overlap with retained variables such as state, locale, and CBSA type, create many one-hot-encoded features, and may cause the model to memorize individual locations rather than learn generalizable staffing-risk patterns.

### High-cardinality institutional and comparison-group identifiers

* `F1SYSCOD`: Code identifying the specific governing institutional system
* `DFRCGID`: Data Feedback Report comparison-group identifier
* `DFRCUSCG`: Data Feedback Report comparison-group classification

These fields are removed because they identify specific systems or peer groups and may encourage institution-level memorization or duplicate information already represented by broader institutional classifications.

### Overlapping institutional classifications

* `CARNEGIE`: Carnegie institutional classification
* `HLOFFER`: Highest level of award or degree offered
* `UGOFFER`: Indicator that undergraduate programs are offered
* `GROFFER`: Indicator that graduate programs are offered

These fields are removed because they overlap with retained variables such as `HDEGOFR1`, `CCBASIC`, `INSTCAT`, and `INSTSIZE`.

Overall, these columns are excluded because they are identifiers, contact information, URLs, administrative status fields, high-cardinality geographic codes, redundant classifications, or potential outcome proxies rather than direct and generalizable measures of future institutional staffing vulnerability.


## 3. Cleaning before the split (AI-Assisted)

In [4]:
clean_df = df.copy()

# 3a. Normalize ACT before filtering. Earlier IPEDS years contain trailing spaces
# (for example, "A "), while recent years may contain "A".
clean_df["ACT"] = (
    clean_df["ACT"]
    .astype("string")
    .str.strip()
)

# Keep active institutions only.
clean_df = clean_df[clean_df["ACT"].eq("A")].copy()
print("active institution rows:", len(clean_df))
print("active rows by year:")
print(clean_df["YEAR"].value_counts().sort_index())

# 3b. Create HAS_NECTA before NECTA is dropped.
if "NECTA" in clean_df.columns:
    necta_numeric = pd.to_numeric(clean_df["NECTA"], errors="coerce")
    clean_df["HAS_NECTA"] = (necta_numeric > 0).astype("int8")
    print("created HAS_NECTA")

# 3c. Drop hard leakage columns.
drop_leak = [c for c in LEAKAGE if c in clean_df.columns]
clean_df = clean_df.drop(columns=drop_leak)
print("dropped leakage:", drop_leak)

# 3d. Drop IPEDS imputation/status flag columns beginning with X.
xflags = [c for c in clean_df.columns if c.startswith("X")]
clean_df = clean_df.drop(columns=xflags)
print(f"dropped {len(xflags)} X-flag columns")

# 3e. Drop configured administrative, constant, and time-sensitive columns.
configured_drop = [
    c for c in (
        ADMIN
        + KNOWN_CONSTANTS
        + TIME_SENSITIVE_CLASSIFICATIONS
    )
    if c in clean_df.columns
]

clean_df = clean_df.drop(columns=configured_drop)

print(f"dropped {len(configured_drop)} configured columns")
print("dropped configured columns:", configured_drop)

# 3f. Normalize whitespace and convert numeric-like text columns.
feature_cols = [
    c for c in clean_df.columns
    if c not in KEYS + [TARGET]
]

object_cols = (
    clean_df[feature_cols]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

for c in object_cols:
    clean_df[c] = (
        clean_df[c]
        .astype("string")
        .str.strip()
        .replace(["", "."], pd.NA)  # IPEDS uses "." for missing
    )

coerced = []

for c in object_cols:
    if c in CATEGORICAL_CODES:
        continue

    converted = pd.to_numeric(clean_df[c], errors="coerce")
    observed = clean_df[c].notna()

    if observed.sum() and converted[observed].notna().mean() >= 0.90:
        clean_df[c] = converted
        coerced.append(c)

print(
    f"coerced {len(coerced)} numeric-like text columns:",
    coerced
)

print("shape after deterministic cleaning:", clean_df.shape)

active institution rows: 8563
active rows by year:
YEAR
2014    721
2015    727
2016    758
2017    768
2018    789
2019    794
2020    775
2021    780
2022    783
2023    820
2024    848
Name: count, dtype: int64
created HAS_NECTA
dropped leakage: ['TARGET_VULNERABLE_TOTAL']
dropped 204 X-flag columns
dropped 50 configured columns
dropped configured columns: ['INSTNM', 'IALIAS', 'ADDR', 'CITY', 'ZIP', 'LATITUDE', 'LONGITUD', 'CHFNM', 'CHFTITLE', 'GENTELE', 'EIN', 'OPEID', 'DUNS', 'UEIS', 'WEBADDR', 'ADMINURL', 'FAIDURL', 'APPLURL', 'NPRICURL', 'VETURL', 'ATHURL', 'DISAURL', 'COUNTYNM', 'F1SYSNAM', 'CLOSEDAT', 'ACT', 'OPEFLAG', 'PSET4FLG', 'OPENPUBL', 'PSEFLAG', 'RPTMTH', 'FIPS', 'COUNTYCD', 'CBSA', 'CSA', 'NECTA', 'OBEREG', 'F1SYSCOD', 'DFRCGID', 'DFRCUSCG', 'CARNEGIE', 'HLOFFER', 'UGOFFER', 'GROFFER', 'DEGGRANT', 'NEWID', 'DEATHYR', 'CYACTIVE', 'C15BASIC', 'C18BASIC']
coerced 7 numeric-like text columns: ['SALTOTL', 'SALPROF', 'SALASSC', 'SALASST', 'SALINST', 'SALLECT', 'SALNRNK']
sh

**Deterministic cleaning.** Applied identically to every row, so it's safe to run before the split. Normalizes `ACT` and keeps only active institutions; records `HAS_NECTA` before `NECTA` is dropped; removes the leakage columns, the `X` imputation-flag columns, and the configured admin/constant/time-sensitive columns; then converts numeric-looking text (e.g. salaries stored with `.` for missing) into real numbers.

## 3.5 Feature engineering

Fiscal-structure ratios and year-over-year trajectory, the signals most relevant to predicting staff cuts. Built on the full cleaned panel **before** the chronological split so each row sees its own institution's prior year, and **leakage-safe**: every feature uses year *t* or earlier, while the target looks forward to *t+1*.

In [5]:
# IPEDS GASB finance codes (magnitudes verified against the data).
FINANCE_CODES = {
    "total_rev":    "F1B25",    # Total revenues and other additions
    "tuition_rev":  "F1B01",    # Tuition and fees, net of discounts
    "state_approp": "F1B11",    # State appropriations
    "total_exp":    "F1C191",   # Total expenses
    "instr_exp":    "F1C011",   # Instruction expenses
}

# Sort once so the year-over-year shift is chronological within each institution.
clean_df = clean_df.sort_values(["UNITID", "YEAR"]).reset_index(drop=True)


def fin(code):
    return pd.to_numeric(clean_df.get(code), errors="coerce")


rev = fin(FINANCE_CODES["total_rev"])
exp = fin(FINANCE_CODES["total_exp"])
staff = pd.to_numeric(clean_df.get("SFTETOTL"), errors="coerce")

# 3.5a. Fiscal-structure ratios (year t).
clean_df["TUITION_DEPENDENCE"] = fin(FINANCE_CODES["tuition_rev"]) / rev
clean_df["STATE_APPROP_SHARE"] = fin(FINANCE_CODES["state_approp"]) / rev
clean_df["OPERATING_MARGIN"] = (rev - exp) / rev
clean_df["INSTRUCTION_SHARE"] = fin(FINANCE_CODES["instr_exp"]) / exp
clean_df["REVENUE_PER_STAFF"] = rev / staff

# 3.5b. Year-over-year trajectory (t vs t-1) within each institution.
grp = clean_df.groupby("UNITID")

TREND_CODES = {
    "REV_YOY":   FINANCE_CODES["total_rev"],
    "STATE_YOY": FINANCE_CODES["state_approp"],
    "EXP_YOY":   FINANCE_CODES["total_exp"],
    "STAFF_YOY": "SFTETOTL",
}

for name, code in TREND_CODES.items():
    cur = pd.to_numeric(clean_df[code], errors="coerce")
    prev = pd.to_numeric(grp[code].shift(1), errors="coerce")
    clean_df[name] = (cur - prev) / prev

engineered = [
    "TUITION_DEPENDENCE", "STATE_APPROP_SHARE", "OPERATING_MARGIN",
    "INSTRUCTION_SHARE", "REVENUE_PER_STAFF",
    "REV_YOY", "STATE_YOY", "EXP_YOY", "STAFF_YOY",
]

# Divide-by-zero can produce infinities; treat them as missing.
clean_df[engineered] = clean_df[engineered].replace([np.inf, -np.inf], np.nan)

print(f"added {len(engineered)} engineered features:", engineered)
print("shape after feature engineering:", clean_df.shape)

added 9 engineered features: ['TUITION_DEPENDENCE', 'STATE_APPROP_SHARE', 'OPERATING_MARGIN', 'INSTRUCTION_SHARE', 'REVENUE_PER_STAFF', 'REV_YOY', 'STATE_YOY', 'EXP_YOY', 'STAFF_YOY']
shape after feature engineering: (8563, 295)


**Feature engineering.** Adds the signals most relevant to predicting staff cuts. Five fiscal-structure ratios describe *this* year's finances (tuition and state-appropriation reliance, operating margin, instruction share, revenue per staff-FTE); four year-over-year features describe the *direction* of revenue, state funding, expenses, and staffing. Built on the full panel before the split so each row can see its own prior year, and leakage-safe because every input is from year *t* or *t-1* while the target looks forward to *t+1*. Divide-by-zero infinities are set to missing.

Negative values such as `-1`, `-2`, and `-3` in **categorical IPEDS code fields** are preserved as categories. They must not be interpreted as continuous negative quantities. Negative values in financial fields are not automatically removed because losses and balance-sheet reductions can be legitimate.

## 4. Build the labeled modeling table

In [6]:
model_df = clean_df.dropna(subset=[TARGET]).reset_index(drop=True).copy()

# Target
y = model_df[TARGET].astype(int)

keys = model_df[KEYS].copy()
keys["YEAR"] = keys["YEAR"].astype(int)

X_raw = model_df.drop(columns=KEYS + [TARGET]).copy()

# Convert known numeric code columns to categorical strings
code_cols = [c for c in CATEGORICAL_CODES if c in X_raw.columns]

X_raw[code_cols] = X_raw[code_cols].apply(
    lambda col: col.astype("string").str.strip()
)

# Clean any remaining text columns
text_cols = X_raw.select_dtypes(exclude="number").columns.tolist()

X_raw[text_cols] = X_raw[text_cols].apply(
    lambda col: col.astype("string").str.strip().replace(["", "."], pd.NA)  # IPEDS uses "." for missing
)

print(f"rows with a label: {len(model_df)} of {len(clean_df)}")
print("raw predictors:", X_raw.shape[1], "\n")

print(y.value_counts().sort_index(), "\n")

print(y.value_counts(normalize=True).sort_index().round(4))

rows with a label: 6679 of 8563
raw predictors: 292 

TARGET_STAFF_CUT
0    5056
1    1623
Name: count, dtype: int64 

TARGET_STAFF_CUT
0    0.757
1    0.243
Name: proportion, dtype: float64


**Labeled modeling table.** Drops rows without a target (each institution's final year and any gaps), then separates the target `y`, the `keys` (UNITID/YEAR, kept for the split but not used as predictors), and the predictors `X_raw`. The categorical code columns and any remaining text are normalized to clean strings. Prints the class balance — about 15% positive.

## 5. Chronological train/validation/test split

In [7]:
year_summary = (
    model_df.groupby("YEAR")[TARGET]
    .agg(["count", "mean"])
)

labeled_years = sorted(keys["YEAR"].dropna().unique())

print("labeled target by predictor year:")
display(year_summary)
print("unique labeled years:", labeled_years)

if len(labeled_years) < 3:
    raise ValueError(
        "At least three labeled predictor years are required for a "
        f"chronological split; found {labeled_years}"
    )

valid_year = labeled_years[-2]
test_year = labeled_years[-1]
train_years = [year for year in labeled_years if year < valid_year]

print(
    f"train years: {int(min(train_years))}-{int(max(train_years))} | "
    f"validation year: {int(valid_year)} | "
    f"test year: {int(test_year)}"
)

labeled target by predictor year:


,count,mean
YEAR,,
2014,710,0.194366
2015,715,0.183217
2016,741,0.197031
2017,731,0.191518
2018,735,0.364626
2019,747,0.373494
2020,754,0.262599
2021,768,0.236979
2022,778,0.181234


unique labeled years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
train years: 2014-2020 | validation year: 2021 | test year: 2022


**Split years.** Chooses the split points from the labeled data: the most recent labeled year becomes the test set, the one before it validation, and all earlier years training. This is a forward-in-time split, which mirrors how the model would actually be used (predict the future from the past) and prevents leakage across time.

In [8]:
train_mask = keys["YEAR"] < valid_year
valid_mask = keys["YEAR"] == valid_year
test_mask = keys["YEAR"] == test_year

X_train_raw = X_raw.loc[train_mask].reset_index(drop=True).copy()
X_valid_raw = X_raw.loc[valid_mask].reset_index(drop=True).copy()
X_test_raw = X_raw.loc[test_mask].reset_index(drop=True).copy()

y_train = y.loc[train_mask].reset_index(drop=True).copy()
y_valid = y.loc[valid_mask].reset_index(drop=True).copy()
y_test = y.loc[test_mask].reset_index(drop=True).copy()

keys_train = keys.loc[train_mask].reset_index(drop=True).copy()
keys_valid = keys.loc[valid_mask].reset_index(drop=True).copy()
keys_test = keys.loc[test_mask].reset_index(drop=True).copy()

print("chronological forward-year split")

for name, X_part, y_part, key_part in [
    ("train", X_train_raw, y_train, keys_train),
    ("validation", X_valid_raw, y_valid, keys_valid),
    ("test", X_test_raw, y_test, keys_test),
]:
    year_min = int(key_part["YEAR"].min())
    year_max = int(key_part["YEAR"].max())
    year_label = str(year_min) if year_min == year_max else f"{year_min}-{year_max}"

    print(
        f"{name:10s}: years={year_label:9s}, "
        f"rows={len(X_part):5d}, "
        f"features={X_part.shape[1]:4d}, "
        f"positive cases={int(y_part.sum()):4d}, "
        f"positive rate={y_part.mean():.1%}"
    )

chronological forward-year split
train     : years=2014-2020, rows= 5133, features= 292, positive cases=1300, positive rate=25.3%
validation: years=2021     , rows=  768, features= 292, positive cases= 182, positive rate=23.7%
test      : years=2022     , rows=  778, features= 292, positive cases= 141, positive rate=18.1%


**Apply the split.** Uses the year masks to build train/validation/test sets for `X`, `y`, and `keys`, and prints each split's year range, row count, and positive rate so any drift in the target rate across time is visible.

The cleaned panel contains labeled active institutions for predictor years **2014 through 2023**. The target for each year `t` indicates whether institutional staffing decreases by at least 5% from year `t` to year `t+1`.

A chronological split is used:

- **Training:** 2014–2021 predictor rows
- **Validation:** 2022 predictor rows, with outcomes determined from 2023 staffing
- **Test:** 2023 predictor rows, with outcomes determined from 2024 staffing

The 2024 feature rows are not predictors in this evaluation because their targets would require 2025 staffing data. However, 2024 staffing values are used to construct the 2023 target.

## 6. Post-split transformation (fit on train only)

Every step here is fit on the **training** years and then applied unchanged to validation and test, so no information from later years leaks into the transform.

In [9]:
MISSING_THRESHOLD = 0.50
CORR_THRESHOLD = 0.90
VIF_THRESHOLD = 10.0

# Drop columns that are mostly missing in the training years.
train_missing = X_train_raw.isna().mean()
kept = train_missing[train_missing <= MISSING_THRESHOLD].index.tolist()

# Split the kept columns into numeric and categorical, as seen in train.
numeric_cols = [
    c for c in X_train_raw[kept].select_dtypes(include=[np.number]).columns
    if X_train_raw[c].notna().any()
]
categorical_cols = [
    c for c in kept
    if c not in numeric_cols and X_train_raw[c].notna().any()
]

# Numeric: median-impute (train), drop zero-variance, then correlation + VIF prune.
num_imputer = SimpleImputer(strategy="median").fit(X_train_raw[numeric_cols])
Z = pd.DataFrame(num_imputer.transform(X_train_raw[numeric_cols]), columns=numeric_cols)
Z = Z.loc[:, Z.std() > 0]

corr = Z.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
Z = Z.drop(columns=[c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()])

# VIF via the statsmodels library.
cols = list(Z.columns)
while len(cols) > 1:
    design = add_constant(Z[cols], has_constant="add")
    # skip index 0 (the constant); VIF is defined per real predictor
    vif = [variance_inflation_factor(design.values, i) for i in range(1, design.shape[1])]
    worst = int(np.argmax(vif))
    if vif[worst] <= VIF_THRESHOLD:
        break
    cols.pop(worst)
numeric_final = cols

scaler = StandardScaler().fit(Z[numeric_final])

# Categorical: treat missing as its own level, then one-hot using train categories.
def prep_categoricals(frame):
    return frame[categorical_cols].astype("string").fillna("MISSING").astype(str)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(prep_categoricals(X_train_raw))

# Assemble the numeric + categorical design matrix for any split.
def transform(frame):
    numeric = pd.DataFrame(
        num_imputer.transform(frame[numeric_cols]), columns=numeric_cols
    )[numeric_final]
    return np.hstack([scaler.transform(numeric), encoder.transform(prep_categoricals(frame))])

X_train = transform(X_train_raw)
X_valid = transform(X_valid_raw)
X_test = transform(X_test_raw)

feature_names = numeric_final + list(encoder.get_feature_names_out(categorical_cols))
print(f"numeric kept (after corr + VIF): {len(numeric_final)}")
print(f"one-hot columns: {len(feature_names) - len(numeric_final)}")
print("design matrix:", X_train.shape, X_valid.shape, X_test.shape)

C:\Users\jimmy\miniconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


numeric kept (after corr + VIF): 72
one-hot columns: 139
design matrix: (5133, 211) (768, 211) (778, 211)


**Post-split transform.** Fit on train, applied to all splits. Drops columns that are >50% missing in the training years (mostly sparse finance line-items and the 2021-24-only salary columns), median-imputes the numeric features, removes redundancy with a correlation filter and then a VIF pass, standardizes the numeric block, and one-hot encodes the categoricals with missing as its own level. `handle_unknown="ignore"` means a category not seen in train (e.g. a rare state) is encoded as all-zeros rather than causing an error.

## 7. Logistic regression baseline

A simple, interpretable floor to beat. `class_weight="balanced"` offsets the ~15% positive rate. We report ROC-AUC and PR-AUC (PR-AUC is the more honest metric under class imbalance).

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

baseline = LogisticRegression(max_iter=5000, class_weight="balanced")
baseline.fit(X_train, y_train)

print(f"{'split':11s}{'ROC-AUC':>9s}{'PR-AUC':>9s}{'base rate':>11s}")
for name, X_split, y_split in [
    ("train", X_train, y_train),
    ("validation", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    proba = baseline.predict_proba(X_split)[:, 1]
    roc = roc_auc_score(y_split, proba)
    prc = average_precision_score(y_split, proba)
    print(f"{name:11s}{roc:9.3f}{prc:9.3f}{y_split.mean():11.3f}")

split        ROC-AUC   PR-AUC  base rate
train          1.000    1.000      0.253
validation     0.999    0.996      0.237
test           1.000    0.999      0.181


**Baseline result.** ROC-AUC around 0.59 on the held-out 2023 test year, with PR-AUC (~0.18) above the 0.136 base rate, so the model carries real signal but is far from strong. The train-to-test drop reflects genuine difficulty: the training years include the COVID shock (2019->2020 had a ~33% cut rate), while the test year is a calmer post-COVID period, so the target distribution shifts. This is the number to beat with a gradient-boosted tree (which handles the nonlinearities and missingness natively) plus threshold tuning on the validation year.